In [ ]:
!pip install -U transformers

## Local Inference on GPU
Model page: https://huggingface.co/google/flan-t5-base

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/google/flan-t5-base)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [1]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [2]:
import pandas as pd

df = pd.read_csv("summarized_reviews.csv")

print(df.columns.tolist())
print(f"Total rows: {len(df)}")

['original_review', 'original_word_count', 'summary', 'summary_word_count']
Total rows: 10


In [3]:

question_reviews = df[df["original_review"].astype(str).str.contains(r"\?", regex=True, na=False)]

if not question_reviews.empty:
    selected_review = question_reviews.iloc[0]["original_review"]
    print("Selected review with question mark:\n")
    print(selected_review)
else:
    print("No review with '?' found. Trying a question-like pattern...")

    question_patterns = r"\b(how|why|what|when|where|can|could|would|should|is it|does it|do you)\b"
    question_reviews = df[df["original_review"].astype(str).str.contains(question_patterns, case=False, regex=True, na=False)]

    if not question_reviews.empty:
        selected_review = question_reviews.iloc[0]["original_review"]
        print("Selected review with question-like wording:\n")
        print(selected_review)
    else:
        raise ValueError("No question-type review found in summarized_reviews.csv")

Selected review with question mark:

I have had Dreamweaver MX2004 since it came out back then. Spent years with it. Feel like I know it real well, but I am still familiar with tables as opposed to CSS. So I thought this would be a great introduction, and it is. The problem is that while I am looking at the video, and that simplifies things a lot, I am not getting the intuitive explanation as to why things work the way they do. I understand just knowing how to work them is sufficient. I'm tempted to delve into the rich full attributes of Dreamweaver CS5 as explained by this video, albeit difficult to understand the more advanced features, but I won't because this is about the video.

The opening salvo is chock full of... for example, this is the URL; this is where you type in the address bar etc. Only when you start to get into areas such as CSS do you get an introduction that for me is beneficial, and that is only because I am somewhat of a newbie to it. So there you have it, do you g

In [4]:

prompt = f"""
You are a customer service representative.

Read the customer review below and write a polite and helpful response.
Start with an apology, address the customer's concern clearly, and suggest a practical solution.

Customer review:
{selected_review}

Response:
"""

print(prompt)


You are a customer service representative.

Read the customer review below and write a polite and helpful response.
Start with an apology, address the customer's concern clearly, and suggest a practical solution.

Customer review:
I have had Dreamweaver MX2004 since it came out back then. Spent years with it. Feel like I know it real well, but I am still familiar with tables as opposed to CSS. So I thought this would be a great introduction, and it is. The problem is that while I am looking at the video, and that simplifies things a lot, I am not getting the intuitive explanation as to why things work the way they do. I understand just knowing how to work them is sufficient. I'm tempted to delve into the rich full attributes of Dreamweaver CS5 as explained by this video, albeit difficult to understand the more advanced features, but I won't because this is about the video.

The opening salvo is chock full of... for example, this is the URL; this is where you type in the address bar et

In [5]:

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)


outputs = model.generate(
    **inputs,
    max_new_tokens=120,
    num_beams=4,
    temperature=0.7,
    early_stopping=True
)

generated_response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Generated service response:\n")
print(generated_response)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Generated service response:

I'm sorry to hear that Dreamweaver MX2004 doesn't have a tutorial. I'm sorry to hear that. I'm sorry to hear that Dreamweaver MX2004 doesn't have a tutorial.


In [6]:

result_df = pd.DataFrame({
    "selected_review": [selected_review],
    "generated_response": [generated_response]
})

result_df.to_csv("task17_response.csv", index=False)
print("Saved to task17_response.csv")

Saved to task17_response.csv
